In [18]:
import behaviors
import no_signaling_sets
import numpy as np
from tqdm import tqdm

## Parameters

In [19]:
delta=2
m=2

n_hyperplanes = int(1e5)

## Hyperplane Extraction and Analysis

### One sample plane

In [20]:
non_srns_samples = np.load("../data/non_srns/non_srns_points.npy")
# non_srns_samples = list(non_srns_samples)

example_sample = np.random.choice(len(non_srns_samples), 1)[0]
example_sample = non_srns_samples[example_sample]
example_sample = behaviors.RoutedBehavior(delta, m, vector=example_sample)

srns_set = no_signaling_sets.ShortRangeNoSignalingSet(delta, m)

print(f"Example sample: {example_sample}")
print(f"Example sample is no-signaling: [{example_sample.is_no_signaling()}]")

np.linalg.matrix_rank(np.array(non_srns_samples))

Example sample: Behavior:
Short path (z=S):
[[0.10505486 0.17843757 0.2222696  0.30768587]
 [0.11952387 0.04614115 0.34071125 0.25529498]
 [0.37168627 0.43685296 0.25447153 0.30760467]
 [0.403735   0.33856831 0.18254762 0.12941448]]
Long path (z=L) :
[[0.00371246 0.03651929 0.52060745 0.29773007]
 [0.22086627 0.18805944 0.04237339 0.26525078]
 [0.52423292 0.62422914 0.00733792 0.36301836]
 [0.25118836 0.15119214 0.42968123 0.0740008 ]]
------------
Example sample is no-signaling: [True]


np.int64(15)

### Some rank estimation & stuff

In [21]:
family = non_srns_samples[:n_hyperplanes]
family_rank = np.linalg.matrix_rank(family)
print(f"Rank of family: {family_rank}")

chosen_hyperplane = srns_set.get_facet_hyperplane(example_sample)
hyperplane_behavior = behaviors.RoutedBehavior(delta, m, vector=chosen_hyperplane)

print(f"Hyperplane: {chosen_hyperplane}")
print(f"In behavior expression: {hyperplane_behavior}")

2025-05-26 11:37:49.003 | WARNING  | no_signaling_sets:get_facet_hyperplane:384 - The has rank 11, not yet identified as maximal hyperplane dimension.


Rank of family: 15
Hyperplane: [ 0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          1.76064774  0.
  0.          0.          1.76064774 -1.76064774  0.          0.
  0.          0.          0.          0.          1.76064774  0.
 -1.76064774  1.76064774]
In behavior expression: Behavior:
Short path (z=S):
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
Long path (z=L) :
[[ 1.76064774  0.          0.          0.        ]
 [ 1.76064774 -1.76064774  0.          0.        ]
 [ 0.          0.          0.          0.        ]
 [ 1.76064774  0.         -1.76064774  1.76064774]]
------------


## Bulk process samples for all hyperplanes

In [22]:
def scale_down_vector(vector):
    mask = np.abs(vector) > 1e-10

    factor = np.abs(vector[mask])[0]

    abs_is_cst = np.allclose(np.abs(vector[mask]), factor, atol=1e-10)
    # print(f"{np.abs(vector[mask])} == {factor} ? {abs_is_cst}")
    if abs_is_cst:
        # Divide by the factor and round to nearest integer to avoid floating point issues
        rescaled = np.zeros_like(vector)
        rescaled[mask] = np.round(vector[mask] / factor).astype(int)
        return rescaled
    else:
        raise ValueError(f"Vector cannot be scaled down uniformly : {vector}")

def normalize_vector(vector):
    if np.allclose(vector, 0, atol=1e-10):
        return vector
    else:
        return vector / np.linalg.norm(vector)

all_hyperplanes_scaled = set()
all_hyperplanes_normalized = set()

In [23]:
i = 0
for vec in list(non_srns_samples)[:n_hyperplanes]:
    _, _, vec_lambda = srns_set.is_facet_hyperplane(
        behaviors.RoutedBehavior(delta, m, vec)
        )
    rescaled_vec = scale_down_vector(vec_lambda)

    if str(rescaled_vec) not in all_hyperplanes_scaled:
        all_hyperplanes_scaled.add(str(rescaled_vec))
        i += 1
        print(f"[{i}] {behaviors.RoutedBehavior(delta, m, rescaled_vec[:32])}")
        # print(f"Hyperplane vector: {rescaled_vec}")

for vec in tqdm(list(non_srns_samples)[:n_hyperplanes]):
    _, _, vec_lambda = srns_set.is_facet_hyperplane(
        behaviors.RoutedBehavior(delta, m, vec)
        )
    normalized_vec = normalize_vector(vec_lambda)

    all_hyperplanes_normalized.add(str(normalized_vec))

[1] Behavior:
Short path (z=S):
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
Long path (z=L) :
[[ 1.  0. -1.  0.]
 [ 0.  0.  0.  1.]
 [ 1.  0.  0.  0.]
 [ 1. -1.  0.  0.]]
------------
[2] Behavior:
Short path (z=S):
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
Long path (z=L) :
[[ 0.  0.  1.  0.]
 [ 0.  0.  1. -1.]
 [-1.  0.  1.  0.]
 [ 0.  1.  0.  0.]]
------------
[3] Behavior:
Short path (z=S):
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
Long path (z=L) :
[[ 0.  0.  1.  0.]
 [ 0.  0.  0.  0.]
 [-1.  0.  1.  0.]
 [ 0.  1.  1. -1.]]
------------
[4] Behavior:
Short path (z=S):
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
Long path (z=L) :
[[-1.  0.  1.  0.]
 [ 0.  1.  0.  0.]
 [ 0.  0.  1.  0.]
 [ 0.  0.  1. -1.]]
------------
[5] Behavior:
Short path (z=S):
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
Long path (z=L) :
[[-1.  0.  1.  0.]
 [ 0.  1.  1. -1.]
 [ 0.  0.  1.  0.]
 [ 0.  0.  0.  0.]]
--------

100%|██████████| 100000/100000 [04:14<00:00, 392.83it/s]


In [44]:
import re

print(f"Number of hyperplanes scaled: {len(all_hyperplanes_scaled)}")
copy_hyperplanes = list(all_hyperplanes_scaled.copy())
copy_hyperplanes.sort()
for hyperplane in copy_hyperplanes:
    printable = hyperplane.replace("\n", "").replace(".", "")

    regex_1 = re.sub(r"(\d)\s(\d)", r'\1  \2', printable)
    regex_2 = re.sub(r"(\d)\s(\d)", r'\1  \2', regex_1)
    final = re.sub(r"(\[)(\d)", r'\1 \2', regex_2)

    print(final)

print(f"Number of hyperplanes normalized: {len(all_hyperplanes_normalized)}")
copy_hyperplanes_normalized = list(all_hyperplanes_normalized.copy())
copy_hyperplanes_normalized.sort()
for hyperplane in copy_hyperplanes_normalized:
    printable = hyperplane.replace("\n", "")

    regex_1 = re.sub(r"0\.(\d{7,11})", r'1.', printable)
    regex_2 = re.sub(r"(\d\.)\s\s+(\d\.)", r'\1 \2', regex_1)
    regex_2 = re.sub(r"(\d\.)\s\s+(\d\.)", r'\1 \2', regex_2)
    regex_3 = re.sub(r"(\d)\.\s(\d\.)", r'\1  \2', regex_2)
    regex_3 = re.sub(r"(\d)\.\s(\d)", r'\1  \2', regex_3)
    regex_4 = re.sub(r"(\d)\.\s+(\-\d)", r'\1 \2', regex_3)
    final = re.sub(r"(\d)\s*(\])", r'\1\2', regex_4.replace(".", ""))

    print(final)

Number of hyperplanes scaled: 32
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  1  0  1 -1  0  0  0  1  0  0]
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  1  0  0  0  0 -1  1  0  0  0  0  1]
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  0  1  0 -1  1  1 -1  0  0  1  0]
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  1  0  0  0  0 -1  0  1  0  0  0  1]
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  0  0  1  0  1  0  0 -1  0  0  0 -1]
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  1  0  0  0  0  0  1 -1  0 -1  0  0]
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  1 -1  1  0  0  0  0  0  0  0  0 -1  0  0]
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  1  0  0  1  0 -1  0  0  0 -1]
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  0  1  0  0  0  1

In [40]:
A, b = srns_set.get_equations(example_sample)

with np.printoptions(precision=1, threshold=np.inf):
    for line in A[-4:,1:]:
        print(str(line).replace(".", "").replace("\n", ""))
    # print(str().replace(".", ""))

[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1 -1  0  0  0  0  0  0  1 -1  0  0  0  0  0  0]
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1 -1  0  0  0  0  0  0  1 -1  0  0  0  0]
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1 -1  0  0  0  0  0  0  1 -1  0  0]
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1 -1  0  0  0  0  0  0  1 -1]


## With (2,2,2) hyperplanes in a file, do more work on it